# Sparse Walker: dynamic memory-bank hypothesis test

One experiment, one kill criterion. The local Walker stays **K=8, degree=4, 2 hops**. It gains a dynamic bank of at most **16 historical sparse states**. A state is written only when novel; the current state routes to the **top-2 bank states** and continues walking from their concepts.

## Hypotheses and diagnostics

1. **Addressable old states fix long-history loss.** Success target: val NDCG@10 >= **0.145**.
2. **The bank actually causes the gain.** Counterfactual: evaluate the same trained model with memory reads disabled.
3. **The gain is long-range.** Report NDCG delta for <=50, 51-100, and 101-200 event histories.
4. **The model really jumps backward.** Report mean retrieval age and fractions >=32 / >=64 events old.
5. **Memory is dynamic rather than full-history attention.** Report novelty-write rate and final bank occupancy.
6. **Systems cost stays sparse.** A one-batch speed gate aborts if the bank is >2.5x plain Walker.


In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, sys, subprocess, shutil, torch
REPO='/content/Sparsewalker'
BRANCH='agent/walker-memory-bank'
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(['git','clone','-q','-b',BRANCH,'https://github.com/hanialshater/Sparsewalker-.git',REPO],check=True)
SRC=f'{REPO}/src'
sys.path.insert(0,SRC)
for name in list(sys.modules):
    if name=='sparsewalker' or name.startswith('sparsewalker.'):
        del sys.modules[name]
from sparsewalker.models import SparseWalkerMemoryBank
from sparsewalker.models.core import ar_training_loss
assert torch.cuda.is_available(), 'GPU runtime required'
print('GPU',torch.cuda.get_device_name(0),'bf16',torch.cuda.is_bf16_supported())
m=SparseWalkerMemoryBank(3706,200,d=64,layers=2,side=256,h=16,active=8,top_side=2,degree=4,bank_size=16,memory_topk=2,initial_memory_share=.10).cuda().train()
x=torch.randint(1,3707,(4,41),device='cuda')
with torch.autocast('cuda',dtype=torch.bfloat16):
    H=m.encode(x[:,:-1]); loss=ar_training_loss(m,x,loss_mode='full')
loss.backward()
print('SMOKE OK',tuple(H.shape),'loss',float(loss.detach()))
del m,x,H,loss
torch.cuda.empty_cache()


## Run the falsification experiment

This runs **in the notebook kernel** (no subprocess buffering). It prints the speed gate first, then every 5 training batches. Maximum 8 warm-start fine-tuning epochs; it stops earlier if the hypothesis is clearly weak or if NDCG reaches 0.145.

In [ ]:
import runpy, sys
SCRIPT=f'{REPO}/experiments/run_ml1m_memory_bank.py'
sys.argv=[SCRIPT,'--seed','42','--max-epochs','8','--batch-size','128','--eval-batch-size','1024','--success-ndcg','0.145']
runpy.run_path(SCRIPT,run_name='__main__')


## Inspect saved result

In [ ]:
import json, pandas as pd
from pathlib import Path
root=Path('/content/drive/MyDrive/sparsewalker_memory_bank/ml1m/seed42')
hp=root/'history.csv'; rp=root/'result.json'
if hp.exists(): display(pd.read_csv(hp))
if rp.exists(): print(json.dumps(json.loads(rp.read_text()),indent=2))
